# Интерактивный конспект — StyleCLIP (Часть 2 задания 23.1)

> Тема: **редактирование изображений по текстовому описанию** через оптимизацию
> латента StyleGAN2.
> Базовая статья: [StyleCLIP: Text-Driven Manipulation of StyleGAN Imagery](https://arxiv.org/abs/2103.17249)
> (Patashnik et al., 2021).

## Задача

Дано лицо, сгенерированное StyleGAN2. Нужно изменить его согласно тексту
(«человек с зелёными волосами», «улыбающийся человек», «с бородой`»), **сохранив
узнаваемость человека**. Пар «было / стало» для обучения нет — текст и изображение
относятся к разным модальностям.

## Идея StyleCLIP (вариант latent optimization)

- **StyleGAN2** генерирует реалистичные лица из латентного вектора; малый сдвиг
  латента → правдоподобное изменение картинки.
- **CLIP** измеряет, насколько изображение «похоже» на текст → задаёт направление.
- Объединив их, ставим **задачу оптимизации**: ищем латент `w`, при котором
  сгенерированное изображение максимально близко к промпту по CLIP.

Берём исходный латент `w_s`, делаем его обучаемой переменной и градиентным спуском
минимизируем составной лосс. Веса всех сетей (StyleGAN2, CLIP, ArcFace) **заморожены** —
оптимизируется только сам вектор `w`.

```
текст t  ──► CLIP text encoder ──┐
                                 ├─► CLIP loss ──┐
w  ──► StyleGAN2 ──► image G(w) ─┘               │
 │                      │                        ├─► общий loss ──► backward ──► ∇w
 │                      └─► ArcFace ──► ID loss ──┤
 └────────────────► ||w - w_s||₂ ──► L2 loss ─────┘
```

Этот конспект **самодостаточен**: тяжёлые модели (StyleGAN2 / CLIP / ArcFace) не
загружаются — все концепции демонстрируются на лёгкой синтетике с `torch`, `numpy`,
`matplotlib`.

In [ ]:
# Лёгкие библиотеки — никаких StyleGAN2/CLIP/ArcFace
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

print("torch:", torch.__version__)
print("Сиды зафиксированы (42) — конспект воспроизводим.")

## 1. Латентные пространства StyleGAN2: Z → W

StyleGAN2 генерирует изображение не напрямую из шума, а через несколько латентных
пространств.

| Пространство | Что это | Размерность (FFHQ) | Свойства |
|---|---|---|---|
| **Z** | Исходный шум `z ~ N(0, I)` | 512 | Распределение фиксировано (гауссиана), «запутанное» (entangled). |
| **W** | Выход mapping-сети `w = MLP(z)` | 512 | Не обязан быть гауссовым, **расцеплённое** (disentangled). |
| **W+** | Свой `w` на каждый из 18 слоёв генератора | 18 × 512 | Самое выразительное, но легко «вылетает» из реалистичного многообразия. |

**Mapping-сеть** — 8-слойный MLP, преобразующий простую гауссиану `Z` в сложно
устроенное пространство `W`. В коде задания: `latent_w = generator.style(latent_z)`.

**Почему оптимизируют в W:** W расцеплено (маленький сдвиг чаще меняет один атрибут),
один вектор 512 чисел компактен, оптимизация устойчива, картинка остаётся реалистичной.
Z слишком запутан, W+ даёт артефакты из-за избыточной свободы.

Ниже — синтетическая mapping-сеть `z → w`, чтобы увидеть, что нелинейный MLP меняет
распределение латента.

In [ ]:
# Игрушечная mapping-сеть z -> w (аналог 8-слойного MLP StyleGAN2, но крохотная)
class ToyMapping(nn.Module):
    def __init__(self, dim=2, depth=4):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers += [nn.Linear(dim, dim), nn.LeakyReLU(0.2)]
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

mapping = ToyMapping(dim=2, depth=4)

# Z ~ N(0, I) — простая гауссиана
z = torch.randn(1000, 2)
with torch.no_grad():
    w = mapping(z)   # W = MLP(z) — распределение «искривлено»

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(z[:, 0], z[:, 1], s=8, alpha=0.4, color="steelblue")
ax[0].set_title("Z: исходный шум ~ N(0, I)")
ax[1].scatter(w[:, 0], w[:, 1], s=8, alpha=0.4, color="indianred")
ax[1].set_title("W = MLP(z): искривлённое расцепленное пространство")
for a in ax:
    a.set_aspect("equal"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Mapping-сеть превращает простую гауссиану Z в нетривиально устроенное W.")

## 2. Косинусное сходство и косинусное расстояние

Близость двух эмбеддингов измеряется **косинусным сходством** — косинусом угла между
векторами (масштаб не важен, важно направление):

$$\text{cos\_sim}(a, b) = \frac{\langle a, b\rangle}{\lVert a\rVert\cdot\lVert b\rVert}$$

Значения: `+1` — одинаковое направление, `0` — ортогональны, `-1` — противоположны.

**Косинусное расстояние** = `1 - cos_sim` — именно оно служит лоссом: `0` — векторы
совпадают по направлению, `2` — противоположны. Минимизация `1 - cos` поворачивает
один вектор к другому.

In [ ]:
# Косинусное сходство и расстояние на torch
def cosine_similarity(a, b, eps=1e-8):
    a_n = a / (a.norm(dim=-1, keepdim=True) + eps)
    b_n = b / (b.norm(dim=-1, keepdim=True) + eps)
    return (a_n * b_n).sum(dim=-1)

def cosine_distance(a, b):
    return 1.0 - cosine_similarity(a, b)   # это и есть loss

a = torch.tensor([[1.0, 0.0]])
for label, b in [("совпадает",      torch.tensor([[2.0,  0.0]])),
                  ("под 45°",       torch.tensor([[1.0,  1.0]])),
                  ("ортогонален",   torch.tensor([[0.0,  1.0]])),
                  ("противоположен",torch.tensor([[-1.0, 0.0]]))]:
    sim = cosine_similarity(a, b).item()
    print(f"{label:16s}: cos_sim = {sim:+.3f},  1 - cos = {1 - sim:.3f}")

# проверка против эталона torch
assert torch.allclose(cosine_similarity(a, torch.tensor([[1.0, 1.0]])),
                      torch.nn.functional.cosine_similarity(a, torch.tensor([[1.0, 1.0]])))
print("\nСовпадает с torch.nn.functional.cosine_similarity.")

## 3. CLIP-style loss на синтетических эмбеддингах

**CLIP** (Contrastive Language-Image Pre-training) — два энкодера (image и text),
которые выдают векторы в **одном общем пространстве**. Поэтому картинку и текст можно
сравнивать напрямую косинусом.

**CLIP loss** = косинусное расстояние между эмбеддингом изображения и эмбеддингом текста:

$$D_{\text{CLIP}}(G(w), t) = 1 - \text{cos\_sim}\big(E_{\text{img}}(G(w)),\; E_{\text{text}}(t)\big)$$

Ниже — синтетика: «текстовый» вектор фиксирован, «картиночный» вектор оптимизируется.
Видно, как минимизация `1 - cos` поворачивает image-эмбеддинг к text-эмбеддингу.

In [ ]:
# «Текстовый» эмбеддинг фиксирован, «картиночный» — обучаемый
text_emb = torch.tensor([[1.0, 1.0]])                 # цель (промпт)
img_emb  = torch.tensor([[1.0, -0.6]], requires_grad=True)  # старт

opt = torch.optim.Adam([img_emb], lr=0.1)
trajectory = [img_emb.detach().clone()]
losses = []
for step in range(60):
    opt.zero_grad()
    loss = cosine_distance(img_emb, text_emb)   # CLIP-style loss
    loss.backward()
    opt.step()
    losses.append(loss.item())
    trajectory.append(img_emb.detach().clone())

traj = torch.cat(trajectory)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses, color="indianred")
ax[0].set_title("CLIP-style loss = 1 - cos_sim"); ax[0].set_xlabel("шаг")
ax[0].grid(alpha=0.3)
ax[1].plot(traj[:, 0], traj[:, 1], "-o", ms=3, color="steelblue", label="траектория img-эмб.")
ax[1].arrow(0, 0, text_emb[0, 0], text_emb[0, 1], color="green",
            head_width=0.08, length_includes_head=True, label="text-эмб.")
ax[1].set_title("img-эмбеддинг поворачивается к text-эмбеддингу")
ax[1].set_aspect("equal"); ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.show()

print(f"CLIP-style loss: {losses[0]:.4f} -> {losses[-1]:.4f}")

## 4. L2-регуляризация латента

StyleGAN2 выдаёт реалистичные лица только для латентов из «обжитой» области W. Если
`w` уйдёт слишком далеко, картинка пойдёт артефактами. Поэтому добавляют **L2 loss** —
штраф за отклонение от исходного латента `w_s`:

$$L_2 = \lVert w - w_s\rVert_2$$

На шаге 0 `w == w_s`, поэтому `L2 = 0`; по мере редактирования он закономерно растёт —
это «цена» правки. L2-штраф удерживает `w` рядом с `w_s` (в реалистичной зоне) и
заставляет делать **минимальную** правку.

Эксперимент ниже: один и тот же CLIP-сигнал, но разный вес штрафа `λ₂` — больший `λ₂`
сильнее притормаживает уход латента.

In [ ]:
# Как lambda_2 сдерживает уход латента от w_s
w_s = torch.tensor([[0.0, 0.0]])           # исходный латент (постоянный эталон)
target_direction = torch.tensor([[3.0, 3.0]])  # куда «тянет» CLIP-сигнал

def run(l2_lambda, steps=120):
    w = w_s.detach().clone().requires_grad_(True)
    opt = torch.optim.Adam([w], lr=0.1)
    dist_hist = []
    for _ in range(steps):
        opt.zero_grad()
        clip_like = cosine_distance(w + 1e-6, target_direction)  # тянет к цели
        l2 = torch.norm(w - w_s)                                  # штраф
        loss = clip_like + l2_lambda * l2
        loss.backward()
        opt.step()
        dist_hist.append(torch.norm(w - w_s).item())
    return dist_hist

plt.figure(figsize=(7, 4))
for lam in [0.0, 0.05, 0.2, 0.6]:
    h = run(lam)
    plt.plot(h, label=f"lambda_2 = {lam}  (финал ||w-w_s|| = {h[-1]:.2f})")
plt.xlabel("шаг"); plt.ylabel("||w - w_s||  (L2 loss)")
plt.title("Больший lambda_2 сильнее удерживает латент возле w_s")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. ID loss — сохранение идентичности

Если оптимизировать только CLIP loss, латент может «уплыть» так, что под промпт
подгонится **другое лицо**. Чтобы этого не было, добавляют **ID loss**.

**ArcFace** — сеть распознавания лиц (в задании ResNet-50 + SE-блоки, `ir_se`, вход
112×112). Обучена с Additive Angular Margin Loss: разные люди — далеко на гиперсфере,
одно лицо в разных кадрах — близко. Backbone заканчивается `l2_norm` → выдаёт уже
нормированные эмбеддинги.

**ID loss** — косинусное расстояние между эмбеддингом исходного и редактируемого лица:

$$L_{\text{ID}}(w) = 1 - \langle R(G(w_s)),\; R(G(w))\rangle$$

Чем ближе к 0, тем лучше сохранена личность. Эмбеддинг исходного лица берётся с
`.detach()` (постоянный эталон).

In [ ]:
# Синтетика: «эмбеддинг лица» = маленькая фиксированная сеть.
# Это НЕ настоящий ArcFace — лишь демонстрация идеи ID loss.
face_net = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 4))
for p in face_net.parameters():
    p.requires_grad_(False)            # сеть заморожена, как ArcFace

def id_embed(latent):
    e = face_net(latent)
    return e / (e.norm(dim=-1, keepdim=True) + 1e-8)   # l2_norm, как в ArcFace

w_src = torch.tensor([[0.5, -0.3]])
ref_embed = id_embed(w_src).detach()    # эталон идентичности, .detach()

for shift, name in [(0.0, "тот же латент"),
                     (0.4, "небольшая правка"),
                     (1.5, "сильный уход")]:
    w_edit = w_src + shift
    id_loss = 1.0 - (id_embed(w_edit) * ref_embed).sum()
    print(f"{name:18s}: ID loss = {id_loss.item():.4f}")

print("\nЧем дальше латент уходит — тем больше ID loss (личность теряется).")

## 6. Полный лосс и мини-демо оптимизации

StyleCLIP (latent optimization) минимизирует по `w` сумму трёх слагаемых:

$$\arg\min_w \; D_{\text{CLIP}}(G(w), t)\;+\;\lambda_2\cdot\lVert w - w_s\rVert_2\;+\;\lambda_{\text{ID}}\cdot L_{\text{ID}}(w)$$

| Слагаемое | Роль | За что отвечает |
|---|---|---|
| `D_CLIP` | **главный** лосс | Двигает изображение к тексту. |
| `λ₂·‖w - w_s‖₂` | регуляризация | Реалистичность, минимальная правка. |
| `λ_ID·L_ID` | регуляризация | Сохранение идентичности по ArcFace. |

В задании `λ₂ = 0.008`, `λ_ID = 0.001` — оба малы, CLIP loss доминирует, L2 и ID
работают как **мягкая регуляризация**. Ниже — мини-демо: оптимизируем синтетический
«латент» Adam'ом по полному лоссу и строим график падения.

In [ ]:
# Мини-демо полного лосса StyleCLIP на синтетике
L2_LAMBDA = 0.008      # как в задании
ID_LAMBDA = 0.001      # как в задании

w_s2     = torch.tensor([[0.2, 0.1]])              # исходный латент
text_dir = torch.tensor([[2.5, -2.0]])             # «текстовое» направление CLIP

latent = w_s2.detach().clone().requires_grad_(True)  # оптимизируем ТОЛЬКО латент
opt = torch.optim.Adam([latent], lr=0.08)

hist = {"clip": [], "l2": [], "id": [], "all": []}
ref = id_embed(w_s2).detach()
for step in range(200):
    opt.zero_grad()
    clip_loss = cosine_distance(latent + 1e-6, text_dir)
    l2_loss   = torch.norm(latent - w_s2)
    id_loss   = 1.0 - (id_embed(latent) * ref).sum()
    loss = clip_loss + L2_LAMBDA * l2_loss + ID_LAMBDA * id_loss
    loss.backward()
    opt.step()
    for k, v in zip(hist, [clip_loss, l2_loss, id_loss, loss]):
        hist[k].append(v.item())

plt.figure(figsize=(7, 4))
for k, c in zip(["clip", "l2", "id", "all"], ["indianred", "steelblue", "green", "black"]):
    plt.plot(hist[k], label=k, color=c)
plt.yscale("symlog")
plt.xlabel("шаг"); plt.ylabel("loss (symlog)")
plt.title("Полный лосс StyleCLIP: clip + l2_lambda*l2 + id_lambda*id")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"CLIP loss: {hist['clip'][0]:.4f} -> {hist['clip'][-1]:.4f}")
print(f"L2   loss: {hist['l2'][0]:.4f} -> {hist['l2'][-1]:.4f}  (растёт — цена правки)")
print(f"ID   loss: {hist['id'][0]:.4f} -> {hist['id'][-1]:.4f}")
print(f"Сумма    : {hist['all'][0]:.4f} -> {hist['all'][-1]:.4f}")

## 7. ExponentialLR — затухание learning rate

В задании используется `ExponentialLR(optimizer, gamma=0.99)`: на каждом шаге
`lr ← lr · gamma`. При старте `lr = 0.08` после 200 шагов получится
`0.08 · 0.99²⁰⁰ ≈ 0.0107` — крупные правки в начале, тонкая подстройка в конце.

In [ ]:
# Кривая learning rate для ExponentialLR(gamma=0.99), 200 шагов
dummy = torch.nn.Parameter(torch.zeros(1))
opt_lr = torch.optim.Adam([dummy], lr=0.08)
sched  = torch.optim.lr_scheduler.ExponentialLR(opt_lr, gamma=0.99)

lrs = []
for step in range(200):
    lrs.append(opt_lr.param_groups[0]["lr"])
    opt_lr.step()
    sched.step()

plt.figure(figsize=(7, 4))
plt.plot(lrs, color="darkorange")
plt.xlabel("шаг"); plt.ylabel("learning rate")
plt.title("ExponentialLR (gamma=0.99): lr_t = 0.08 * 0.99^t")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"lr на шаге 0  : {lrs[0]:.6f}")
print(f"lr на шаге 199: {lrs[-1]:.6f}")
print(f"проверка 0.08 * 0.99^200 = {0.08 * 0.99 ** 200:.6f}")

## 8. Градиент ко входу при замороженных весах

Суть StyleCLIP: веса всех сетей **заморожены**, обучается только латент. Как это
работает технически?

- `param.requires_grad_(False)` отключает обновление **весов**, но **не рвёт граф** —
  градиент всё равно течёт **через** слой к его входу (к латенту).
- `torch.no_grad()` рвёт граф полностью — внутри него `input.grad` не появится.
  Для StyleCLIP `no_grad` на пути `latent → … → loss` **недопустим**.

Ниже — `nn.Linear` с замороженными весами: `input.grad` считается, `weight.grad` — нет.

In [ ]:
# Заморожен слой, но градиент ко ВХОДУ всё равно течёт
layer = nn.Linear(4, 3)
for p in layer.parameters():
    p.requires_grad_(False)          # веса заморожены — как в StyleGAN2/CLIP/ArcFace

x = torch.randn(1, 4, requires_grad=True)   # «латент» — обучаемый вход
y = layer(x)
loss = y.pow(2).sum()
loss.backward()

print("=== requires_grad_(False) на весах ===")
print("x.grad             :", None if x.grad is None else "посчитан, форма " + str(tuple(x.grad.shape)))
print("layer.weight.grad  :", layer.weight.grad, "(веса заморожены — градиента нет)")
print("=> граф НЕ порван: градиент дошёл до входа, веса не обновляются.\n")

# Контраст: torch.no_grad() рвёт граф
x2 = torch.randn(1, 4, requires_grad=True)
with torch.no_grad():
    y2 = layer(x2)
print("=== внутри torch.no_grad() ===")
print("y2.requires_grad   :", y2.requires_grad, "(граф оборван)")
print("=> backward() от y2 невозможен, x2.grad останется None.")
print("   Поэтому в StyleCLIP no_grad на пути latent -> loss НЕДОПУСТИМ.")

## 9. Результаты задания (Часть 2)

Промпт: `a person with green hair`. Гиперпараметры: `lr=0.08`,
`ExponentialLR gamma=0.99`, `λ₂=0.008`, `λ_ID=0.001`, `num_steps=200`.

### Динамика лоссов за 200 шагов

| Лосс | Шаг 0 | Финал (шаг 199) | Изменение |
|---|---|---|---|
| **CLIP loss** | 0.7773 | 0.5430 | снизился на **30.2 %** |
| **L2 loss** | 0.0000 | 7.5017 | вырос (латент отошёл от `w_s`) |
| **ID loss** | 0.0365 | 0.5664 | вырос (идентичность дрейфует) |
| **Суммарный (all)** | 0.7774 | 0.6035 | снизился |

Суммарный loss каждые 50 шагов: `0.7774 → 0.6644 → 0.6168 → 0.6233 → финал 0.6035`.

### Интерпретация

- **CLIP loss** в целом снижается `0.7773 → 0.5430` — изображение сдвинулось к тексту
  в CLIP-пространстве (на лице появляется зелёный оттенок волос). До нуля не доходит —
  это нормально: косинусная близость реального лица и короткой фразы ограничена сверху,
  важна **относительная** динамика.
- **L2 loss** растёт от 0 (на шаге 0 `latent == latent_w`) до 7.5017 — цена правки.
- **ID loss** заметно растёт (0.0365 → 0.5664): косинусная близость ArcFace-эмбеддингов
  упала с ~0.96 до ~0.43 — идентичность ощутимо дрейфует. Малый `λ_ID=0.001` почти не
  сдерживает оптимизацию (вклад `λ_ID·L_ID ≈ 0.0006` ничтожен).
- **Суммарный loss** повторяет форму CLIP loss — `λ₂` и `λ_ID` малы, L2/ID работают
  как мягкая регуляризация.

## 10. Три варианта StyleCLIP из статьи

В оригинальной работе предложено три способа редактирования. В задании реализован
только первый.

| Вариант | Как работает | Скорость | Когда применять |
|---|---|---|---|
| **Latent optimization** (реализован) | Для каждой картинки/промпта — отдельный градиентный спуск по латенту (200 шагов). | Медленно: оптимизация заново на каждый запрос. | Произвольный текст, разовое редактирование. |
| **Latent mapper** | Обучается отдельная сеть-маппер: по латенту сразу выдаёт нужный сдвиг под фиксированный промпт. | Быстро на инференсе, но маппер надо обучить. | Один тип правки для многих изображений. |
| **Global directions** | Заранее находится одно универсальное направление в латенте для текстового атрибута; правка = сдвиг вдоль него. | Очень быстро, без оптимизации на инференсе. | Атрибут известен заранее, нужна интерактивность. |

### Ключевые выводы

- **StyleCLIP соединяет три замороженные сети**: StyleGAN2 генерирует, CLIP задаёт
  направление по тексту, ArcFace охраняет идентичность. Обучаемый объект — только латент.
- **Оптимизация в W**: расцеплено и устойчиво, латент держится в реалистичной зоне.
- **CLIP loss = 1 − косинусное сходство**; важна относительная динамика, не ноль.
- **Регуляризаторы обязательны**: без L2 латент уходит в нереалистичную зону, без ID
  loss редактирование может подменить лицо.
- **Сквозная дифференцируемость** — суть метода: любой `no_grad` на пути
  `latent → … → loss` ломает оптимизацию.

## 11. Попробуй сам

Ниже — площадка для экспериментов. Подсказки в комментариях.

In [ ]:
# --- Эксперимент 1: подбери lambda_2 и lambda_id ---
# Поменяй коэффициенты и посмотри, как меняются финальные L2/ID лоссы.
# Подсказка: больший lambda_2 -> меньше финальный L2, но слабее правка под текст.
MY_L2_LAMBDA = 0.008    # попробуй 0.0, 0.05, 0.3
MY_ID_LAMBDA = 0.001    # попробуй 0.0, 0.1, 0.5

w0   = torch.tensor([[0.2, 0.1]])
goal = torch.tensor([[2.5, -2.0]])
lat  = w0.detach().clone().requires_grad_(True)
opt  = torch.optim.Adam([lat], lr=0.08)
ref0 = id_embed(w0).detach()
for _ in range(200):
    opt.zero_grad()
    loss = (cosine_distance(lat + 1e-6, goal)
            + MY_L2_LAMBDA * torch.norm(lat - w0)
            + MY_ID_LAMBDA * (1.0 - (id_embed(lat) * ref0).sum()))
    loss.backward()
    opt.step()
print(f"lambda_2={MY_L2_LAMBDA}, lambda_id={MY_ID_LAMBDA} -> "
      f"финальный L2 = {torch.norm(lat - w0).item():.4f}")

# --- Эксперимент 2: меняй gamma у ExponentialLR ---
# Подсказка: gamma=1.0 -> lr постоянный; gamma=0.95 -> быстрое затухание.
MY_GAMMA = 0.99         # попробуй 1.0, 0.95, 0.999
d = torch.nn.Parameter(torch.zeros(1))
o = torch.optim.Adam([d], lr=0.08)
s = torch.optim.lr_scheduler.ExponentialLR(o, gamma=MY_GAMMA)
for _ in range(200):
    o.step(); s.step()
print(f"gamma={MY_GAMMA} -> lr после 200 шагов = {o.param_groups[0]['lr']:.6f}")

# --- Эксперимент 3 (для размышления) ---
# Оберни forward-проход синтетического лосса в `with torch.no_grad():`
# и попробуй вызвать loss.backward(). Какую ошибку выдаст torch и почему?